[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **Smart Mask Detection System**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

# Mask Detection with Persistent Tracking (YOLOv11x)

## Project Overview  
This project uses **YOLOv11x** (the most powerful version of the YOLOv11 family) to solve a common problem in safety monitoring: **"flickering" detections**.  
Most systems fail to consistently detect masks when a person briefly turns their head or is partially occluded.

To address this, the system integrates **Object Tracking** and **Sticky Logic**:
- Each person is assigned a **unique tracking ID**
- The system **remembers mask status over time**
- Once a mask is detected within a **buffer zone around the head**, the person is marked as:
  - **Masked (Persistent State)**

This ensures reliable, stable, and flicker-free safety monitoring across the entire video clip.

---

## Real-World Applications  

### Hospital & Healthcare Monitoring  
Ensures doctors, nurses, and visitors follow hygiene protocols in sterile environments like ICUs and operation theaters without manual supervision.

### High-Security Construction Sites  
Tracks workers in hazardous environments and triggers alerts if someone enters a "Red Zone" without proper protective gear.

### Public Transport Hubs  
Monitors compliance in crowded areas like airports and metro stations, where people move quickly and frequently turn away from cameras.

### Food Processing Factories  
Acts as an automated quality controller to enforce safety rules such as wearing masks and hairnets.

### Smart Office Entry  
Integrates with automated access systems where doors open only when mask compliance is detected.

---

## Key Features  

- **Persistent Tracking**  
  Uses unique IDs to track individuals consistently, even during movement or occlusion.

- **Sticky Logic System**  
  Prevents flickering by locking mask status once detected.

- **Buffer Zone Technology**  
  Expands the detection area around the head to capture masks more accurately.

- **High Accuracy**  
  Built on **YOLOv11x** for robust, real-time detection performance.

---

## Annotate your Custom dataset using Labellerr

 ***1. Visit the [Labellerr](https://www.labellerr.com/?utm_source=githubY&utm_medium=social&utm_campaign=github_clicks) website and click **“Sign Up”**.*** 

 ***2. After signing in, create your workspace by entering a unique name.***

 ***3. Navigate to your workspace’s API keys page (e.g., `https://<your-workspace>.labellerr.com/workspace/api-keys`) to generate your **API Key** and **API Secret**.***

 ***4. Store the credentials securely, and then use them to initialise the SDK or API client with `api_key`, `api_secret`.*** 



## Import Libraries

This section imports all the required libraries used throughout the project for computer vision, visualization, deep learning, and structured coding.


In [4]:
!pip install ultralytics opencv-python matplotlib cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for cv2


In [10]:
%pip install pyyaml

     -------------------------------------- 158.6/158.6 kB 4.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


## 🎞️ Random Frame Extraction from Video

Extracts a fixed number of high-quality frames from one or more videos to create an image dataset for annotation and training.

### 🔹 Purpose
- Convert raw manufacturing videos into individual image frames  
- Perform random sampling to avoid frame bias  
- Prepare data for annotation and YOLO training  


In [6]:
from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['People Mask Vid Dataset.mp4'],
    total_images=30,
    out_dir="dataset_frames",
    jpg_quality=100,
    seed=42
)

[✓] Extracted 30 frames to folder: dataset_frames


## 📥 Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


# COCO to YOLO Format Conversion

Converts COCO-style segmentation annotations to YOLO segmentation dataset format.  
- Requires: `annotation.json` and images in `frames_output` directory.
- Output: Generated YOLO dataset folder.
- Parameters: allows train/val split, shuffling, and verbose mode.


In [15]:
from yolo_finetune_utils.coco_yolo_converter.seg_converter import coco_to_yolo_converter

coco_to_yolo_converter(
    json_path="3b96d0db-9554-4eae-b6cf-c0e5f743bb02.json",
    images_dir="dataset_frames",
    output_dir="yolo_dataset",
    use_split=True,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    shuffle=True,
    verbose=True
)


Conversion complete. Stats: {'train': 24, 'val': 3, 'test': 3}


{'stats': {'train': 24, 'val': 3, 'test': 3}, 'output_dir': 'yolo_dataset'}

# Load and Train YOLO Segmentation Model

Loads the YOLO segmentation model and trains it using the converted YOLO dataset.
- Data: Path to YOLO-style `data.yaml`
- Parameters: epochs, image size, batch size, device, dataloader workers, experiment name.


In [ ]:
# Initialize the Extra Large model
model = YOLO('yolo11x-seg.pt') 

# Start training
results = model.train(
    data='/kaggle/working/mask_data.yaml',
    epochs=100,
    imgsz=640,
    batch=4,
    device=0, # Uses the Kaggle GPU
    name='head_mask_tracking',
    project='/kaggle/working/runs',
    exist_ok=True # Overwrites if the folder already exists
)

# Inference


## Model Loading  
`model = YOLO(model_path)` loads the custom-trained weights (`best.pt`).  
This enables the system to detect specific classes such as "heads" and "masks".

---

## Smart Tracking  
`model.track(...)` runs the inference with object tracking enabled.

- `stream=True`  
  Processes the video frame-by-frame, reducing VRAM usage.

- `retina_masks=True`  
  Generates high-resolution segmentation masks for better visual accuracy.

- `show_boxes=False`, `show_labels=False`  
  Disables default annotations so custom logic (e.g., Green/Red status) can be applied.

---

In [ ]:
from ultralytics import YOLO
import torch
import gc

# 1. Force clear any existing VRAM leakage
gc.collect()
torch.cuda.empty_cache()

# 2. Load the model
model_path = '/kaggle/working/runs/head_mask_tracking/weights/best.pt'
model = YOLO(model_path)

# 3. Define Video Path
video_input = '/kaggle/input/datasets/aaryanaggarwal5040/videod/People Mask Vid Dataset.mp4'

# 4. Run Inference with Generator to stop at 500 frames
# stream=True is MANDATORY here to prevent RAM/VRAM accumulation
results = model.track(
    source=video_input,
    conf=0.15,
    iou=0.45,
    show=False,
    save=True,
    show_boxes=False,
    show_labels=False,
    project='/kaggle/working/inference',
    name='results',
    retina_masks=True, # Re-enabled as requested
    device=0,
    stream=True
)

# Iterate and break at 500
for i, r in enumerate(results):
    if i >= 500:
        break
    
    # Optional: Periodically clear cache every 50 frames to prevent fragmentation
    if i % 50 == 0:
        torch.cuda.empty_cache()

print(f"✅ Clean inference of first 500 frames (with Retina Masks) complete.")

# The Three Core Pillars

## 1. Identity Persistence (`persist=True`)  
The system does not treat detections as separate objects in every frame.  
Instead, it assigns a **unique ID** to each person, allowing the model to track the same individual across the entire video.

This ensures:
- Continuous tracking even when people move
- Stability during overlaps or partial occlusions

---

## 2. Sticky Logic (`memory` Dictionary)  
This acts as the decision-making layer of the system.

- In standard AI systems:  
  If a mask is not detected in a single frame, the status immediately changes (causing flickering).

- In this system:  
  Once a person is detected wearing a mask, their status is stored as `True` in memory.

Result:
- The person remains classified as **Masked (Green)**  
- Temporary issues like head turns or obstructions do not affect the final output

---

## 3. Spatial Buffer (`BUFFER_PERCENT`)  
A buffer zone is added around the detected head region.

Purpose:
- Expands the detection area slightly beyond the exact bounding box
- Accounts for real-world variations where masks may not perfectly align with the detected head region

This improves:
- Detection reliability  
- Robustness in dynamic environments  

---

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import torch

# 1. Load Model
model = YOLO('/kaggle/working/runs/head_mask_tracking/weights/best.pt')

# 2. Setup Video
video_input = '/kaggle/input/datasets/aaryanaggarwal5040/videod/People Mask Vid Dataset.mp4'
cap = cv2.VideoCapture(video_input)
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
out = cv2.VideoWriter('/kaggle/working/sticky_bbox_project.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# --- CONFIGURATION ---
BUFFER_PERCENT = 0.1  # 25% buffer
MAX_FRAMES = 550
memory = {}  # Dictionary to store {track_id: is_masked_forever}
# ---------------------

# Setting retina_masks=False since we are no longer drawing them
results = model.track(source=video_input, stream=True, retina_masks=False, conf=0.15, device=0, persist=True)

for i, r in enumerate(results):
    if i >= MAX_FRAMES: break
    frame = r.orig_img.copy()
    
    # Check if tracking IDs exist
    if r.boxes is not None and r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        clss = r.boxes.cls.cpu().numpy()
        ids = r.boxes.id.int().cpu().tolist()

        # Step 1: Detect all masks in the current frame
        current_frame_masks = [boxes[j] for j, c in enumerate(clss) if c == 1]

        # Step 2: Process each person (head)
        for j, track_id in enumerate(ids):
            if clss[j] != 0: continue # Skip if not a head (assuming 0=head)
            
            h_box = boxes[j]
            hx1, hy1, hx2, hy2 = h_box

            # Initialize memory for new IDs
            if track_id not in memory:
                memory[track_id] = False

            # If not already marked as masked, check for mask overlap in current frame
            if not memory[track_id]:
                bw, bh = (hx2 - hx1) * BUFFER_PERCENT, (hy2 - hy1) * BUFFER_PERCENT
                ex1, ey1, ex2, ey2 = hx1 - bw, hy1 - bh, hx2 + bw, hy2 + bh
                
                for m_box in current_frame_masks:
                    mx1, my1, mx2, my2 = m_box
                    m_center = ((mx1 + mx2) / 2, (my1 + my2) / 2)
                    if ex1 < m_center[0] < ex2 and ey1 < m_center[1] < ey2:
                        memory[track_id] = True # Mark as masked for the rest of the clip
                        break

            # Step 3: Draw Bounding Box based on Memory
            is_masked = memory[track_id]
            color = (0, 255, 0) if is_masked else (0, 0, 255) # Green if masked, else Red
            label = "Masked" if is_masked else "No Mask"

            # Draw only the Bounding Box and the Label
            cv2.rectangle(frame, (int(hx1), int(hy1)), (int(hx2), int(hy2)), color, 3)
            cv2.putText(frame, label, (int(hx1), int(hy1) - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    out.write(frame)
    if i % 50 == 0: torch.cuda.empty_cache()

cap.release()
out.release()
print("✅ Sticky bounding box project complete!")

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
